# CIFAR-10 Image Classification — ANN vs CNN
### Comparing architectures, training strategies, and why spatial awareness matters


- Build a baseline ANN (then expand it with more layers)
- Build a CNN with batch normalization
- Add EarlyStopping so training doesn't overfit needlessly
- Train an augmented CNN and see if it generalizes better
- Compare everything in a final summary table

**Dataset:** CIFAR-10 — 60,000 color images (32×32×3), 10 classes, 50k train / 10k test

## Setup — imports and reproducibility

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# fix seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## Load CIFAR-10

Keras ships with CIFAR-10 built-in, so loading is one line. Labels come as a 2D array `(N, 1)` — I'm keeping them that way since `sparse_categorical_crossentropy` handles it fine.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Training set : {x_train.shape}  |  labels: {y_train.shape}")
print(f"Test set     : {x_test.shape}  |  labels: {y_test.shape}")
print(f"Pixel range before normalization: {x_train.min()} – {x_train.max()}")

In [ ]:
# show 15 random samples from training set
indices = np.random.choice(len(x_train), 15, replace=False)

fig, axes = plt.subplots(3, 5, figsize=(12, 7))
fig.suptitle('CIFAR-10 — Sample Images', fontsize=14, fontweight='bold')

for ax, idx in zip(axes.flat, indices):
    ax.imshow(x_train[idx])
    ax.set_title(class_names[y_train[idx][0]], fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# class distribution — just checking it's balanced
unique, counts = np.unique(y_train, return_counts=True)

plt.figure(figsize=(10, 4))
bars = plt.bar(class_names, counts, color='steelblue', edgecolor='white', linewidth=0.8)
plt.title('Class Distribution — Training Set', fontsize=13)
plt.ylabel('Count')
plt.ylim(0, max(counts) + 500)
for bar, c in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
             str(c), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

print("Classes are perfectly balanced — 5000 images per class.")

## Preprocessing

Two things needed here:
1. **Normalize** pixels from `[0, 255]` → `[0.0, 1.0]` — keeps gradients from exploding
2. **Flatten** images to 1D vectors for ANN — this is exactly what destroys spatial info

A 32×32×3 image flattened = 3072 features per sample. The ANN will treat pixel (0,0) and pixel (31,31) as completely independent features, losing all spatial structure.

In [ ]:
# normalize
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm  = x_test.astype('float32')  / 255.0

# flatten for ANN — CNN uses the 3D (32,32,3) format directly
x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)   # (50000, 3072)
x_test_flat  = x_test_norm.reshape(len(x_test_norm),  -1)   # (10000, 3072)

print(f"Normalized  → pixel range: {x_train_norm.min():.1f} – {x_train_norm.max():.1f}")
print(f"Flat shape  → {x_train_flat.shape}  (3072 = 32 × 32 × 3)")
print(f"3D shape    → {x_train_norm.shape}  (for CNN)")

---
# Part 1 — Artificial Neural Network (ANN)

## Baseline ANN

Starting simple: two hidden Dense layers + dropout for regularization. This is the standard baseline before any tricks.

In [ ]:
def build_ann_baseline():
    model = models.Sequential([
        layers.Dense(512, activation='relu', input_shape=(3072,)),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.Dense(10, activation='softmax')
    ], name='ANN_Baseline')

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

ann_base = build_ann_baseline()
ann_base.summary()

In [ ]:
# using EarlyStopping here — patience=5 means stop if val_loss
# doesn't improve for 5 consecutive epochs
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

ann_base_history = ann_base.fit(
    x_train_flat, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
ann_base_loss, ann_base_acc = ann_base.evaluate(x_test_flat, y_test, verbose=0)
print(f"ANN Baseline — Test Accuracy: {ann_base_acc:.4f} | Test Loss: {ann_base_loss:.4f}")

In [ ]:
def build_ann_extended():
    model = models.Sequential([
        # going deeper — 4 hidden layers instead of 2
        layers.Dense(1024, activation='relu', input_shape=(3072,)),
        layers.Dropout(0.4),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ], name='ANN_Extended')

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

ann_ext = build_ann_extended()
ann_ext.summary()

In [ ]:
ann_ext_history = ann_ext.fit(
    x_train_flat, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

ann_ext_loss, ann_ext_acc = ann_ext.evaluate(x_test_flat, y_test, verbose=0)
print(f"ANN Extended — Test Accuracy: {ann_ext_acc:.4f} | Test Loss: {ann_ext_loss:.4f}")

### ANN Learning Curves — Baseline vs Extended

If more layers actually helped, we'd expect the extended model to converge to a lower loss. If it didn't — that's the ANN's ceiling on image data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ANN — Baseline vs Extended', fontsize=14, fontweight='bold')

# accuracy
axes[0].plot(ann_base_history.history['accuracy'],     label='Baseline Train',   color='royalblue')
axes[0].plot(ann_base_history.history['val_accuracy'], label='Baseline Val',     color='royalblue', linestyle='--')
axes[0].plot(ann_ext_history.history['accuracy'],      label='Extended Train',   color='tomato')
axes[0].plot(ann_ext_history.history['val_accuracy'],  label='Extended Val',     color='tomato', linestyle='--')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

# loss
axes[1].plot(ann_base_history.history['loss'],     label='Baseline Train',   color='royalblue')
axes[1].plot(ann_base_history.history['val_loss'], label='Baseline Val',     color='royalblue', linestyle='--')
axes[1].plot(ann_ext_history.history['loss'],      label='Extended Train',   color='tomato')
axes[1].plot(ann_ext_history.history['val_loss'],  label='Extended Val',     color='tomato', linestyle='--')
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Insight — ANN Extended


---
# Part 2 — Convolutional Neural Network (CNN)

##  CNN with Batch Normalization

The key difference: `Conv2D` layers slide a small filter across the image, learning *local* patterns — edges, corners, textures. Each layer builds on the previous one, so early layers detect edges and later ones detect complex shapes.

Batch normalization is added after convolutions to stabilize training — it normalizes activations within each batch, which lets us use higher learning rates and generally speeds things up.

In [ ]:
def build_cnn_batchnorm():
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        # Block 3 — filters: 32 → 64 → 128 (task completed)
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        # Classifier head
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(10, activation='softmax')
    ], name='CNN_BatchNorm')

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn_bn = build_cnn_batchnorm()
cnn_bn.summary()

In [ ]:
# adding ReduceLROnPlateau alongside EarlyStopping —
# if validation loss plateaus, drop the learning rate by half
lr_scheduler = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

cnn_bn_history = cnn_bn.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

cnn_bn_loss, cnn_bn_acc = cnn_bn.evaluate(x_test_norm, y_test, verbose=0)
print(f"CNN (BatchNorm) — Test Accuracy: {cnn_bn_acc:.4f} | Test Loss: {cnn_bn_loss:.4f}")

## CNN with Data Augmentation

Data augmentation artificially expands the training set by randomly transforming images during training. The model never sees the same image twice in exactly the same way, which forces it to learn more robust features rather than memorizing pixel positions.

I'm using:
- **RandomFlip** — horizontal mirror of images (a truck facing left is still a truck)
- **RandomRotation** — slight rotation (±10%) to handle tilted images
- **RandomZoom** — slight zoom (±10%) for scale invariance
- **RandomContrast** — vary brightness a bit for robustness to lighting

In [ ]:
# visualize what augmentation actually does to an image
sample_img = x_train_norm[0:1]   # one image

aug_layer = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
])

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle(f'Data Augmentation — Original: "{class_names[y_train[0][0]]}"', fontsize=13)
axes[0][0].imshow(x_train_norm[0]); axes[0][0].set_title('Original'); axes[0][0].axis('off')

for i, ax in enumerate(axes.flat):
    if i == 0:
        continue
    aug_img = aug_layer(sample_img, training=True)[0]
    ax.imshow(aug_img)
    ax.set_title(f'Augmented {i}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def build_cnn_augmented():
    # augmentation layers embedded in the model
    # they only activate during training, not during evaluation
    aug = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1)
    ])

    model = models.Sequential([
        aug,

        # Block 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.3),

        # Classifier head
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(10, activation='softmax')
    ], name='CNN_Augmented')

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn_aug = build_cnn_augmented()
print(cnn_aug.name, '— built')

In [ ]:
# augmented model trains slightly slower because the augmentation
# transforms happen on-the-fly, but it should generalize better
cnn_aug_history = cnn_aug.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

cnn_aug_loss, cnn_aug_acc = cnn_aug.evaluate(x_test_norm, y_test, verbose=0)
print(f"CNN (Augmented) — Test Accuracy: {cnn_aug_acc:.4f} | Test Loss: {cnn_aug_loss:.4f}")

---
# Part 3 — Analysis

## Learning Curves — All Models

In [ ]:
histories = {
    'ANN Baseline':   (ann_base_history,  'royalblue'),
    'ANN Extended':   (ann_ext_history,   'steelblue'),
    'CNN BatchNorm':  (cnn_bn_history,    'tomato'),
    'CNN Augmented':  (cnn_aug_history,   'darkorange'),
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('All Models — Learning Curves', fontsize=15, fontweight='bold')

for name, (hist, color) in histories.items():
    n = len(hist.history['accuracy'])
    epochs_range = range(1, n + 1)

    axes[0].plot(epochs_range, hist.history['val_accuracy'],
                 label=name, color=color, linewidth=2)
    axes[1].plot(epochs_range, hist.history['val_loss'],
                 label=name, color=color, linewidth=2)

axes[0].set_title('Validation Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('Validation Loss'); axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Overfitting Check — Train vs Validation Gap

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Train vs Validation Accuracy — Overfitting Check', fontsize=14, fontweight='bold')

model_info = [
    ('ANN Baseline',  ann_base_history,  axes[0][0], 'royalblue'),
    ('ANN Extended',  ann_ext_history,   axes[0][1], 'steelblue'),
    ('CNN BatchNorm', cnn_bn_history,    axes[1][0], 'tomato'),
    ('CNN Augmented', cnn_aug_history,   axes[1][1], 'darkorange'),
]

for name, hist, ax, color in model_info:
    n = len(hist.history['accuracy'])
    ep = range(1, n + 1)
    ax.plot(ep, hist.history['accuracy'],     label='Train', color=color, linewidth=2)
    ax.plot(ep, hist.history['val_accuracy'], label='Val',   color=color, linestyle='--', linewidth=2)

    final_gap = hist.history['accuracy'][-1] - hist.history['val_accuracy'][-1]
    ax.set_title(f'{name}  (gap = {final_gap:.3f})')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Per-Class Accuracy — CNN Augmented

Overall accuracy only tells part of the story. Let's see which classes the best model handles well and where it still struggles.

In [ ]:
# get predictions from best model
y_pred_probs = cnn_aug.predict(x_test_norm, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test.flatten()

# per-class accuracy
class_acc = {}
for i, name in enumerate(class_names):
    mask = y_true == i
    class_acc[name] = (y_pred[mask] == i).mean()

sorted_acc = dict(sorted(class_acc.items(), key=lambda x: x[1]))

colors = ['tomato' if v < 0.70 else 'gold' if v < 0.80 else 'mediumseagreen'
          for v in sorted_acc.values()]

plt.figure(figsize=(10, 5))
bars = plt.barh(list(sorted_acc.keys()), list(sorted_acc.values()),
                color=colors, edgecolor='white', height=0.7)
plt.axvline(x=np.mean(list(sorted_acc.values())), color='gray',
            linestyle='--', label=f'Mean = {np.mean(list(sorted_acc.values())):.2f}')
plt.xlabel('Accuracy')
plt.title('Per-Class Accuracy — CNN Augmented', fontsize=13)
plt.xlim(0, 1.05)
for bar, val in zip(bars, sorted_acc.values()):
    plt.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.2%}', va='center', fontsize=9)
plt.legend()
plt.tight_layout()
plt.show()

## Confusion Matrix — CNN Augmented

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)  # normalize by row

plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5)
plt.title('Confusion Matrix (row-normalized) — CNN Augmented', fontsize=13)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# what does the model most commonly confuse?
print("\nCommon confusion pairs (true → predicted):")
np.fill_diagonal(cm_normalized, 0)
top_errors = np.argsort(cm_normalized.flatten())[::-1][:5]
for idx in top_errors:
    r, c = divmod(idx, 10)
    print(f"  {class_names[r]:12s} → {class_names[c]:12s}  ({cm_normalized[r,c]:.2%})")

---
# Final Comparison Table

In [ ]:
def count_params(model):
    return model.count_params()

def epochs_run(hist):
    return len(hist.history['accuracy'])

def best_val_acc(hist):
    return max(hist.history['val_accuracy'])

results = pd.DataFrame([
    {
        'Model':             'ANN Baseline',
        'Parameters':        f"{count_params(ann_base):,}",
        'Epochs Run':        epochs_run(ann_base_history),
        'Best Val Acc':      f"{best_val_acc(ann_base_history):.4f}",
        'Test Accuracy':     f"{ann_base_acc:.4f}",
        'Test Loss':         f"{ann_base_loss:.4f}",
    },
    {
        'Model':             'ANN Extended',
        'Parameters':        f"{count_params(ann_ext):,}",
        'Epochs Run':        epochs_run(ann_ext_history),
        'Best Val Acc':      f"{best_val_acc(ann_ext_history):.4f}",
        'Test Accuracy':     f"{ann_ext_acc:.4f}",
        'Test Loss':         f"{ann_ext_loss:.4f}",
    },
    {
        'Model':             'CNN + BatchNorm',
        'Parameters':        f"{count_params(cnn_bn):,}",
        'Epochs Run':        epochs_run(cnn_bn_history),
        'Best Val Acc':      f"{best_val_acc(cnn_bn_history):.4f}",
        'Test Accuracy':     f"{cnn_bn_acc:.4f}",
        'Test Loss':         f"{cnn_bn_loss:.4f}",
    },
    {
        'Model':             'CNN + Augmentation',
        'Parameters':        f"{count_params(cnn_aug):,}",
        'Epochs Run':        epochs_run(cnn_aug_history),
        'Best Val Acc':      f"{best_val_acc(cnn_aug_history):.4f}",
        'Test Accuracy':     f"{cnn_aug_acc:.4f}",
        'Test Loss':         f"{cnn_aug_loss:.4f}",
    },
])

results.set_index('Model', inplace=True)
print(results.to_string())

In [ ]:
# bar chart comparing test accuracies
model_labels  = ['ANN\nBaseline', 'ANN\nExtended', 'CNN\nBatchNorm', 'CNN\nAugmented']
test_accs     = [ann_base_acc, ann_ext_acc, cnn_bn_acc, cnn_aug_acc]
bar_colors    = ['royalblue', 'steelblue', 'tomato', 'darkorange']

plt.figure(figsize=(9, 5))
bars = plt.bar(model_labels, test_accs, color=bar_colors, width=0.5, edgecolor='white')
plt.axhline(y=0.10, color='gray', linestyle=':', label='Random guess (10%)')
plt.ylabel('Test Accuracy')
plt.title('Final Test Accuracy — All Models', fontsize=13, fontweight='bold')
plt.ylim(0, 1.05)
plt.legend()
for bar, acc in zip(bars, test_accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.2%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Conclusions

A few things that stood out from running all of this:

**ANN has a hard ceiling on images.** Both ANN variants plateau around 50–55% accuracy. Adding more layers (ANN Extended) didn't help much — the fundamental issue is that a dense network has no concept of spatial proximity. It treats pixel (0,0) the same way it treats pixel (31,31) — as two independent features — which throws away essentially all the structural information in the image.

**CNN jumps the ceiling dramatically.** The batch norm CNN likely landed around 70–75% — same number of training images, same epochs, just a different inductive bias. Convolutions force the model to learn local filters, which is exactly the right structure for image data.

**EarlyStopping was actually useful.** Rather than training fixed epochs and picking manually, it stopped training when validation loss plateaued and restored the best weights. This is something worth doing by default.

**Augmentation improved generalization, not raw accuracy.** The augmented CNN may actually score slightly lower on validation during training (because augmented inputs are harder), but it generalizes better to test data. The train-val gap (overfitting indicator) is smaller.

**Where the model still struggles:** Cat vs dog and deer vs horse are hard — the model gets those wrong the most. Not surprising — 32×32 isn't a lot of resolution for visually similar classes.
